In [ ]:
## Working Version
import pandas as pd
import gurobipy as gp
from gurobipy import GRB
# Load the data
data = pd.read_csv('/Users/chris/Library/CloudStorage/OneDrive-UW/3. Junior/MATH 381/Group Project/theater_data.csv')
# Extract necessary parameters
performance_categories = data['Type'].unique()
zones = data['Zone'].unique()
pops = data['popularity'].unique()

# Function to define and optimize the model for a given popularity level
def optimize_for_popularity(popularity_level):
    # Define the Gurobi model
    model = gp.Model('theater_pricing')

    # Define decision variables: price and sales for each performance category and zone
    prices = model.addVars(performance_categories, zones, name="prices", vtype=GRB.CONTINUOUS)
    sales = model.addVars(performance_categories, zones, name="sales", vtype=GRB.CONTINUOUS)

    # Extract price bounds from historical data
    price_bounds = data.groupby(['Type', 'Zone'])['Price'].agg(['min', 'max']).reset_index()
    pij_min = price_bounds.pivot(index='Type', columns='Zone', values='min')
    pij_max = price_bounds.pivot(index='Type', columns='Zone', values='max')

    # Define the demand function based on the given coefficients including popularity
    def demand_expr(price, zone2, zone3, popularity_level):
        if popularity_level == 1:
            pop_2 = 0
            pop_3 = 0
        elif popularity_level == 2:
            pop_2 = 1
            pop_3 = 0
        elif popularity_level == 3:
            pop_2 = 0
            pop_3 = 1
        return 57.38933 + (-0.04035 * price) + (243.97924 * zone2) + (123.91874 * zone3) + (52.74 * pop_2) + (127.24 * pop_3) 

    # Define the objective function: Maximize Profit
    profit = gp.quicksum(sales[category, zone] * prices[category, zone]
                         for category in performance_categories for zone in zones)

    model.setObjective(profit, GRB.MAXIMIZE)

    # Add constraints
    # Constraint 1: Each performance type should not exceed 1200 tickets sold
    for category in performance_categories:
        category_demand = gp.quicksum(sales[category, zone] for zone in zones)
        model.addConstr(category_demand <= 1200, f"category_capacity_{category}")

    # Constraint 2: Ensure each performance series has more than 0 sales
    min_sales_threshold = 1  # You can set this to a higher number if needed
    for category in performance_categories:
        category_sales = gp.quicksum(sales[category, zone] for zone in zones)
        model.addConstr(category_sales >= min_sales_threshold, f"min_sales_{category}")

    # Constraint 4: Sales must match the demand function
    for category in performance_categories:
        for zone in zones:
            zone2_flag = 1 if zone == 2 else 0
            zone3_flag = 1 if zone == 3 else 0
            model.addConstr(sales[category, zone] == demand_expr(prices[category, zone], zone2_flag, zone3_flag, popularity_level), f"demand_{category}_{zone}")

    # Constraint 5: Price bounds
    for category in performance_categories:
        for zone in zones:
            model.addConstr(prices[category, zone] >= pij_min.loc[category, zone], f"min_price_{category}_{zone}")
            model.addConstr(prices[category, zone] <= pij_max.loc[category, zone], f"max_price_{category}_{zone}")

    # Constraint 6: Price hierarchy among zones for each performance category
    for category in performance_categories:
        if 1 in zones and 2 in zones:
            model.addConstr(prices[category, 1] >= prices[category, 2], f"zone_hierarchy_{category}_1_2")
        if 2 in zones and 3 in zones:
            model.addConstr(prices[category, 2] >= prices[category, 3], f"zone_hierarchy_{category}_2_3")

    # Optimize the model
    model.optimize()

    # Output the optimal prices and sales
    optimal_prices = pd.DataFrame(index=performance_categories, columns=zones)
    optimal_sales = pd.DataFrame(index=performance_categories, columns=zones)
    for category in performance_categories:
        for zone in zones:
            optimal_prices.loc[category, zone] = prices[category, zone].X
            optimal_sales.loc[category, zone] = sales[category, zone].X

    print(f"Optimal Prices for popularity {popularity_level}:")
    print(optimal_prices)
    print(f"Optimal Sales for popularity {popularity_level}:")
    print(optimal_sales)

    # Calculate profit for each performance series
    profits = {}
    for category in performance_categories:
        profit = 0
        for zone in zones:
            profit += optimal_sales.loc[category, zone] * optimal_prices.loc[category, zone]
        profits[category] = profit

    print(f"Profits for each performance series for popularity {popularity_level}:")
    for category, profit in profits.items():
        print(f"{category}: ${profit:.2f}")

# Run the optimization for each popularity level
for pop_level in [1, 2, 3]:
    optimize_for_popularity(pop_level)

['Piano' 'Dance' 'Chamber Music' 'Crossroads' 'Special Events']
[1 2 3]
Gurobi Optimizer version 11.0.1 build v11.0.1rc0 (mac64[arm] - Darwin 23.5.0 23F79)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 65 rows, 30 columns and 110 nonzeros
Model fingerprint: 0xec951bde
Model has 15 quadratic objective terms
Coefficient statistics:
  Matrix range     [4e-02, 1e+00]
  Objective range  [0e+00, 0e+00]
  QObjective range [2e+00, 2e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+03]
Presolve removed 40 rows and 0 columns

Continuous model is non-convex -- solving as a MIP

Presolve removed 60 rows and 24 columns
Presolve time: 0.00s
Presolved: 12 rows, 10 columns, 32 nonzeros
Presolved model has 3 bilinear constraint(s)
Variable types: 10 continuous, 0 integer (0 binary)
Found heuristic solution: objective 183337.97696

Root relaxation: interrupted, 0 iterations, 0.00 seconds (0.00 work units